# Clasificación de familias de caracoles con Transfer Learning

Clasificación de imágenes de caracoles en **6 familias** usando **ResNet50** preentrenado en ImageNet.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPUs detectadas: {len(gpus)}" if gpus else "Sin GPU: se entrenara en CPU")

## 1. Carga del dataset

In [ ]:
import zipfile
import os

archivo_zip = 'CARACOLES.zip'
carpeta_destino = 'mis_imagenes'

# Verificar si el archivo ZIP existe antes de descomprimir
if os.path.exists(archivo_zip):
    print(f"Descomprimiendo {archivo_zip} en '{carpeta_destino}'...")
    try:
        with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
            zip_ref.extractall(carpeta_destino)
        print("¡Descompresión completada con éxito!")
    except zipfile.BadZipFile:
        print("Error: El archivo ZIP está dañado o no es un formato válido.")
else:
    print(f"Error: No se encontró el archivo '{archivo_zip}' en el directorio actual.")


clases = sorted(os.listdir('mis_imagenes/CARACOLES'))
print(f"{len(clases)} clases encontradas: {clases}")

directorio_datos = 'mis_imagenes/CARACOLES'

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

### División 60 % train / 20 % validación / 20 % test (estratificada)

`image_dataset_from_directory` solo permite dos subconjuntos y su reparto no
garantiza proporciones por clase. En su lugar se listan las rutas de las
imágenes y se reparten con `train_test_split` de sklearn usando `stratify`,
así cada familia queda representada en la misma proporción en los tres
subconjuntos.

In [ ]:
from sklearn.model_selection import train_test_split

class_names = clases
num_classes = len(class_names)
print(f"Clases de caracoles encontradas ({num_classes}): {class_names}")

EXTENSIONES_IMG = ('.bmp', '.gif', '.jpeg', '.jpg', '.png')
rutas, etiquetas = [], []
for i, clase in enumerate(class_names):
    carpeta = os.path.join(directorio_datos, clase)
    archivos = [f for f in os.listdir(carpeta) if f.lower().endswith(EXTENSIONES_IMG)]
    rutas += [os.path.join(carpeta, f) for f in archivos]
    etiquetas += [i] * len(archivos)

# 60 % train + 40 % temporal; el temporal se parte en mitades → 20 % val + 20 % test
rutas_train, rutas_rest, y_train, y_rest = train_test_split(
    rutas, etiquetas, test_size=0.4, random_state=123, stratify=etiquetas)
rutas_val, rutas_test, y_val, y_test = train_test_split(
    rutas_rest, y_rest, test_size=0.5, random_state=123, stratify=y_rest)

print(f"Total: {len(rutas)} imágenes → train {len(y_train)} | "
      f"validación {len(y_val)} | test {len(y_test)}")
for i, nombre in enumerate(class_names):
    print(f"  {nombre}: {y_train.count(i)} train / "
          f"{y_val.count(i)} val / {y_test.count(i)} test")

AUTOTUNE = tf.data.AUTOTUNE


def cargar_imagen(ruta, etiqueta):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3,
                             expand_animations=False)
    return tf.image.resize(img, IMG_SIZE), etiqueta


def construir_dataset(rutas, etiquetas, barajar=False):
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if barajar:
        ds = ds.shuffle(len(rutas), seed=123, reshuffle_each_iteration=True)
    return ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)


train_dataset = construir_dataset(rutas_train, y_train, barajar=True)
validation_dataset = construir_dataset(rutas_val, y_val)
test_dataset = construir_dataset(rutas_test, y_test)

conteos_clases = {clase: len(os.listdir(os.path.join(directorio_datos, clase)))
                  for clase in class_names}
for clase, n in conteos_clases.items():
    print(f"  {clase}: {n} imágenes")

### Distribución de imágenes por clase

La clase minoritaria es **Risooinidae**: tiene ~4,5 veces menos imágenes que la mayoritaria (Neritidae), así que el dataset está desbalanceado.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar(conteos_clases.keys(), conteos_clases.values(),
                color='steelblue', edgecolor='black')
media = sum(conteos_clases.values()) / len(conteos_clases)
ax.axhline(media, color='red', linestyle='--', label=f'Media = {media:.0f}')
for barra, v in zip(barras, conteos_clases.values()):
    ax.text(barra.get_x() + barra.get_width() / 2, v, str(v),
            ha='center', va='bottom')
ax.set_title('Distribución de imágenes por clase (dataset completo)')
ax.set_xlabel('Familia')
ax.set_ylabel('Número de imágenes')
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('distribucion_clases.png', dpi=150)
plt.show()

**Optimizar la carga de datos**

In [ ]:
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

### Pesos de clase para compensar el desbalance

`class_weight` multiplica la pérdida de cada muestra según su clase, de modo que los errores en clases minoritarias penalizan más. Se calcula **solo con las etiquetas de entrenamiento** (después del split) para no usar información del conjunto de validación.

In [ ]:
train_labels = np.concatenate([y.numpy() for _, y in train_dataset])
conteos_train = np.bincount(train_labels, minlength=num_classes)
class_weights = {
    i: len(train_labels) / (num_classes * int(n))
    for i, n in enumerate(conteos_train)
}

for i, nombre in enumerate(class_names):
    print(f"  {nombre}: {conteos_train[i]} imágenes en train → peso {class_weights[i]:.3f}")

## 2. Aumento de datos y modelo base

In [ ]:
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.1),
  tf.keras.layers.RandomContrast(0.1),
])

**Preprocesamiento y modelo base**

Cada modelo recibe su **propia instancia de ResNet50**: si compartieran la misma base, el fine-tuning de uno modificaría los pesos del otro y la comparación sería inválida.

In [ ]:
preprocess_input = tf.keras.applications.resnet50.preprocess_input

IMG_SHAPE = IMG_SIZE + (3,)


def construir_modelo(usar_augmentation):
    base_model = tf.keras.applications.ResNet50(input_shape=IMG_SHAPE,
                                                include_top=False,
                                                weights='imagenet')
    # Congelo el modelo base: solo se entrenará la cabeza clasificadora
    base_model.trainable = False

    inputs = tf.keras.Input(shape=IMG_SHAPE)
    x = data_augmentation(inputs) if usar_augmentation else inputs
    x = preprocess_input(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes)(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])
    return model, base_model


def crear_callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                             patience=3,
                                             restore_best_weights=True)]


def aplicar_fine_tuning(model, base_model, ultimas_capas=40):
    # Descongelo solo las últimas capas y bajamos el LR para no
    # destruir los pesos preentrenados
    base_model.trainable = True
    for layer in base_model.layers[:-ultimas_capas]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])
    print(f"Pesos entrenables: {len(model.trainable_weights)}")

## 3. Extracción de características

### Modelo SIN data augmentation

In [ ]:
initial_epochs = 10

print("--- Entrenando Modelo SIN Data Augmentation ---")
model_sin_aug, base_sin = construir_modelo(usar_augmentation=False)
history_sin = model_sin_aug.fit(train_dataset,
                              epochs=initial_epochs,
                              validation_data=validation_dataset,
                              class_weight=class_weights,
                              callbacks=crear_callbacks())

### Modelo CON data augmentation

In [ ]:
print("\n--- Entrenando Modelo CON Data Augmentation ---")
model_con_aug, base_con = construir_modelo(usar_augmentation=True)
history_con = model_con_aug.fit(train_dataset,
                              epochs=initial_epochs,
                              validation_data=validation_dataset,
                              class_weight=class_weights,
                              callbacks=crear_callbacks())

## 4. Fine-tuning

Descongelamos las **últimas 40 capas** de ResNet50 en **ambos** modelos y seguimos entrenando con una tasa de aprendizaje 10× menor.

In [ ]:
fine_tune_epochs = 10
ep_sin = len(history_sin.history['loss'])
aplicar_fine_tuning(model_sin_aug, base_sin)
print("--- Fine-tuning del modelo SIN Data Augmentation ---")
history_fine_sin = model_sin_aug.fit(train_dataset,
                                   epochs=ep_sin + fine_tune_epochs,
                                   initial_epoch=ep_sin,
                                   validation_data=validation_dataset,
                                   class_weight=class_weights,
                                   callbacks=crear_callbacks())

fine_tune_epochs = 10
ep_con = len(history_con.history['loss'])
aplicar_fine_tuning(model_con_aug, base_con)
print("--- Fine-tuning del modelo CON Data Augmentation ---")
history_fine_con = model_con_aug.fit(train_dataset,
                                   epochs=ep_con + fine_tune_epochs,
                                   initial_epoch=ep_con,
                                   validation_data=validation_dataset,
                                   class_weight=class_weights,
                                   callbacks=crear_callbacks())

## 5. Resultados

### Gráficas individuales (train + validación de cada modelo)

In [ ]:
# Historiales completos: extracción de características + fine-tuning
acc_sin = history_sin.history['accuracy'] + history_fine_sin.history['accuracy']
val_acc_sin = history_sin.history['val_accuracy'] + history_fine_sin.history['val_accuracy']
loss_sin = history_sin.history['loss'] + history_fine_sin.history['loss']
val_loss_sin = history_sin.history['val_loss'] + history_fine_sin.history['val_loss']

acc_con = history_con.history['accuracy'] + history_fine_con.history['accuracy']
val_acc_con = history_con.history['val_accuracy'] + history_fine_con.history['val_accuracy']
loss_con = history_con.history['loss'] + history_fine_con.history['loss']
val_loss_con = history_con.history['val_loss'] + history_fine_con.history['val_loss']


def graficar_entrenamiento(train_vals, val_vals, metrica, titulo, archivo, corte_ft):
    plt.figure(figsize=(8, 5))
    plt.plot(train_vals, label='Entrenamiento')
    plt.plot(val_vals, label='Validación')
    plt.axvline(corte_ft - 1, color='gray', linestyle='--', label='Inicio fine-tuning')
    plt.title(titulo)
    plt.xlabel('Épocas')
    plt.ylabel('Precisión' if metrica == 'accuracy' else 'Pérdida')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(archivo, dpi=150)
    plt.show()


# Modelo SIN data augmentation
graficar_entrenamiento(loss_sin, val_loss_sin, 'loss',
                       'Pérdida — modelo SIN data augmentation',
                       'loss_sin_aug.png', ep_sin)
graficar_entrenamiento(acc_sin, val_acc_sin, 'accuracy',
                       'Precisión — modelo SIN data augmentation',
                       'accuracy_sin_aug.png', ep_sin)

# Modelo CON data augmentation
graficar_entrenamiento(loss_con, val_loss_con, 'loss',
                       'Pérdida — modelo CON data augmentation',
                       'loss_con_aug.png', ep_con)
graficar_entrenamiento(acc_con, val_acc_con, 'accuracy',
                       'Precisión — modelo CON data augmentation',
                       'accuracy_con_aug.png', ep_con)

### Matrices de confusión (sobre el conjunto de test, nunca visto en entrenamiento)

In [ ]:
def evaluar_modelo(model, nombre, archivo):
    y_true, y_pred = [], []
    for images, labels in test_dataset:
        logits = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(logits, axis=1))

    print(f"\n=== Reporte de clasificación — modelo {nombre} ===")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicción')
    plt.ylabel('Real')
    plt.title(f'Matriz de confusión — modelo {nombre}')
    plt.tight_layout()
    plt.savefig(archivo, dpi=150)
    plt.show()


evaluar_modelo(model_sin_aug, 'SIN data augmentation', 'matriz_sin_aug.png')
evaluar_modelo(model_con_aug, 'CON data augmentation', 'matriz_con_aug.png')

### Comparación conjunta (ambos modelos en la misma gráfica)

In [ ]:
epocas_sin = range(1, len(loss_sin) + 1)
epocas_con = range(1, len(loss_con) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epocas_sin, val_loss_sin, 'r--', linewidth=2, label='Validación SIN Aumento')
plt.plot(epocas_con, val_loss_con, 'g-', linewidth=2, label='Validación CON Aumento')
plt.plot(epocas_sin, loss_sin, 'r', alpha=0.3, label='Entrenamiento SIN')
plt.plot(epocas_con, loss_con, 'g', alpha=0.3, label='Entrenamiento CON')
plt.title('Comparación de Pérdida (Loss)')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('comparacion_loss.png', dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epocas_sin, val_acc_sin, 'r--', linewidth=2, label='Validación SIN Aumento')
plt.plot(epocas_con, val_acc_con, 'g-', linewidth=2, label='Validación CON Aumento')
plt.plot(epocas_sin, acc_sin, 'r', alpha=0.3, label='Entrenamiento SIN')
plt.plot(epocas_con, acc_con, 'g', alpha=0.3, label='Entrenamiento CON')
plt.title('Comparación de Precisión (Accuracy)')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('comparacion_accuracy.png', dpi=150)
plt.show()

### Ejemplos de predicción (modelo CON data augmentation)

In [ ]:
plt.figure(figsize=(12, 12))
for images, labels in test_dataset.take(1):
    preds = np.argmax(model_con_aug.predict(images, verbose=0), axis=1)
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        pred, real = int(preds[i]), int(labels[i])
        color = 'green' if pred == real else 'red'
        plt.title(f"Pred: {class_names[pred]}\nReal: {class_names[real]}", color=color, fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.savefig('predicciones_ejemplo.png', dpi=150)
plt.show()

model_con_aug.save('clasificador_caracoles.keras')
print("Modelo guardado: clasificador_caracoles.keras")